# Customer Churn & Retention Analytics

**Dataset:** IBM Telco Customer Churn (7,043 customers, 21 columns)  
**Author:** Data Analyst Portfolio  

## Project Overview
This notebook provides end-to-end exploratory analysis, feature engineering, and statistical testing on the Kaggle / IBM Telco Customer Churn dataset.

### STEP 4 — Import Libraries

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

### STEP 5 — Load the Kaggle CSV

In [2]:
df = pd.read_csv(
    "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df.head()


Loaded 7,043 rows and 21 columns.


### STEP 6 — Understand the Dataset Structure

In [3]:
print("Dataset Shape:", df.shape)
print("\nDataset Columns:\n", df.columns.tolist())
print("\nDataset Summary Info:")
df.info()


Dataset Shape: (7043, 21)

Dataset Columns:
 ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Dataset Summary Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 dtypes: float64(1), int64(2), object(18)
memory usage: 1.1+ MB


### STEP 7 — Understand Every Column Group

1. **Customer Demographics:** `customerID`, `gender`, `SeniorCitizen`, `Partner`, `Dependents`  
2. **Subscribed Services:** `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`  
3. **Account & Contract Info:** `tenure`, `Contract`, `PaperlessBilling`, `PaymentMethod`, `MonthlyCharges`, `TotalCharges`  
4. **Target Variable:** `Churn`

### STEP 8 — Check Missing Values & Handle Blank Strings in TotalCharges

In [4]:
print("Null counts prior to conversion:")
print(df.isnull().sum())

# Convert TotalCharges string to numeric float
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print("\nNull counts after to_numeric coercion:")
print(df.isnull().sum())

# Inspect the 11 missing records
missing_total = df[df["TotalCharges"].isnull()]
print(f"\nNumber of records with missing TotalCharges: {len(missing_total)}")
missing_total[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]


Null counts prior to conversion:
TotalCharges: 0 (blank strings present)

Null counts after to_numeric coercion:
TotalCharges: 11

Number of records with missing TotalCharges: 11 (new customers with tenure=0)


### STEP 9 — Check Duplicates

In [5]:
print("Duplicate rows count:", df.duplicated().sum())

if df.duplicated().sum() > 0:
    df = df.drop_duplicates()
    print("Duplicates dropped. Verified count:", df.duplicated().sum())


Duplicate rows count: 0


### STEP 10 — Check Customer IDs Uniqueness

In [6]:
unique_ids = df["customerID"].nunique()
total_records = len(df)

print(f"Unique Customer IDs: {unique_ids}")
print(f"Total Records: {total_records}")
if unique_ids == total_records:
    print("Verified: Each customer has a unique ID.")


Unique Customer IDs: 7043
Total Records: 7043
Verified: Each customer has a unique ID.


### STEP 11 — Convert Churn into Numeric ChurnFlag

In [7]:
df["ChurnFlag"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

df[["Churn", "ChurnFlag"]].head()


### STEP 12 — Calculate Portfolio KPIs

In [8]:
total_customers = df["customerID"].nunique()
print("Total Customers:", total_customers)

churned_customers = df["ChurnFlag"].sum()
print("Churned Customers:", churned_customers)

churn_rate = df["ChurnFlag"].mean() * 100
print(f"Churn Rate: {churn_rate:.2f}%")

retention_rate = 100 - churn_rate
print(f"Retention Rate: {retention_rate:.2f}%")


Total Customers: 7043
Churned Customers: 1869
Churn Rate: 26.54%
Retention Rate: 73.46%


### STEP 13 — Calculate Monthly Revenue at Risk

In [9]:
revenue_at_risk = df.loc[
    df["ChurnFlag"] == 1,
    "MonthlyCharges"
].sum()

print(f"Monthly Revenue at Risk: ${revenue_at_risk:,.2f}")


Monthly Revenue at Risk: $139,130.85


### STEP 14 — EDA Q1: How Many Customers Churned?

In [10]:
churn_counts = df["Churn"].value_counts()
print(churn_counts)

sns.countplot(
    data=df,
    x="Churn"
)

plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.show()


No     5174
Yes    1869
Name: Churn, dtype: int64


### STEP 15 — Churn Rate by Contract Type

In [11]:
contract_churn = (
    df.groupby("Contract")["ChurnFlag"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print(contract_churn)

contract_churn.plot(
    kind="bar"
)

plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=0)
plt.show()


Contract
Month-to-month    42.709677
One year          11.269518
Two year           2.831858
Name: ChurnFlag, dtype: float64


### STEP 16 — Churn Rate by Customer Tenure

In [12]:
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 6, 12, 24, 48, 72],
    labels=[
        "0-6 Months",
        "7-12 Months",
        "13-24 Months",
        "25-48 Months",
        "49-72 Months"
    ]
)

tenure_churn = (
    df.groupby(
        "TenureGroup",
        observed=False
    )["ChurnFlag"]
    .mean()
    .mul(100)
)

print(tenure_churn)

tenure_churn.plot(kind="bar")

plt.title("Churn Rate by Customer Tenure")
plt.xlabel("Tenure Group")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=45)
plt.show()


TenureGroup
0-6 Months      47.438596
7-12 Months     34.221599
13-24 Months    28.653846
25-48 Months    20.077220
49-72 Months     9.248555
Name: ChurnFlag, dtype: float64


### STEP 17 — Churn Rate by Internet Service

In [13]:
internet_churn = (
    df.groupby("InternetService")["ChurnFlag"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print(internet_churn)

internet_churn.plot(kind="bar")

plt.title("Churn Rate by Internet Service")
plt.xlabel("Internet Service")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=0)
plt.show()


InternetService
Fiber optic    41.892766
DSL            18.959108
No              7.405036
Name: ChurnFlag, dtype: float64


### STEP 18 — Monthly Charges vs Churn (Boxplot)

In [14]:
sns.boxplot(
    data=df,
    x="Churn",
    y="MonthlyCharges"
)

plt.title("Monthly Charges by Churn Status")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")
plt.show()


### STEP 19 — Tenure vs Monthly Charges (Scatterplot)

In [15]:
sns.scatterplot(
    data=df,
    x="tenure",
    y="MonthlyCharges",
    hue="Churn",
    alpha=0.6
)

plt.title("Tenure vs Monthly Charges")
plt.xlabel("Tenure (Months)")
plt.ylabel("Monthly Charges")
plt.show()


### STEP 20 — Churn Rate by Payment Method

In [16]:
payment_churn = (
    df.groupby("PaymentMethod")["ChurnFlag"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print(payment_churn)

payment_churn.plot(
    kind="bar"
)

plt.title("Churn Rate by Payment Method")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=45)
plt.show()


PaymentMethod
Electronic check             45.285412
Mailed check                 19.106700
Bank transfer (automatic)    16.709845
Credit card (automatic)      15.243101
Name: ChurnFlag, dtype: float64


### STEP 21 — Churn Rate by Tech Support Status

In [17]:
support_churn = (
    df.groupby("TechSupport")["ChurnFlag"]
    .mean()
    .mul(100)
)

print(support_churn)


TechSupport
No                     41.635470
No internet service     7.405036
Yes                    15.166341
Name: ChurnFlag, dtype: float64


### STEP 22 — Statistics: Welch's t-test on Monthly Charges

In [18]:
churned = df.loc[
    df["ChurnFlag"] == 1,
    "MonthlyCharges"
]

retained = df.loc[
    df["ChurnFlag"] == 0,
    "MonthlyCharges"
]

t_stat, p_value = stats.ttest_ind(
    churned,
    retained,
    equal_var=False
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

if p_value < 0.05:
    print("Result: Statistically significant difference in mean monthly charges between churned and retained customers.")
else:
    print("Result: Insufficient evidence of a difference in mean monthly charges.")


T-statistic: 18.407527632662057
P-value: 8.592446977797746e-73
Result: Statistically significant difference in mean monthly charges between churned and retained customers.


### STEP 23 — Statistics: Chi-Square Test (Contract Type vs Churn)

In [19]:
contract_table = pd.crosstab(
    df["Contract"],
    df["Churn"]
)

print(contract_table)

chi2, p_value, dof, expected = stats.chi2_contingency(
    contract_table
)

print("\nChi-square:", chi2)
print("P-value:", p_value)
print("Degrees of freedom:", dof)

if p_value < 0.05:
    print("\nResult: Statistically significant association between contract type and churn status.")
else:
    print("\nResult: No significant association found.")


Churn             No   Yes
Contract                  
Month-to-month  2220  1655
One year        1307   166
Two year        1647    48

Chi-square: 1184.5965782053753
P-value: 5.863038300673391e-258
Degrees of freedom: 2

Result: Statistically significant association between contract type and churn status.


### STEP 24 — Save Cleaned Dataset

In [20]:
df.to_csv(
    "../data/customer_churn_cleaned.csv",
    index=False
)

print("Cleaned dataset successfully exported to '../data/customer_churn_cleaned.csv'")


Cleaned dataset successfully exported to '../data/customer_churn_cleaned.csv'
